[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrew-l-miller/gwosc/blob/main/make_sfts/make_SFTs.ipynb)

In [ ]:
from pathlib import Path
import os
import sys
import requests
import argparse
import matplotlib.pyplot as plt
import numpy as np
import scipy
import glob
import shlex
import shutil
import subprocess
import re

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    
if IN_COLAB:
    if os.path.basename(os.getcwd()) == "make_sfts":
        print('things already downloaded')
    else:
        print("downloading codes and compiled code")
        !git clone https://github.com/andrew-l-miller/gwosc.git
        os.chdir('gwosc/make_sfts')
    
    
# !wget -nc https://dcc.ligo.org/public/0192/T2400058/003/segsH1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt
# !wget -nc https://dcc.ligo.org/public/0192/T2400058/003/segsL1AnalysisReadyMinusVetoes_O4a_C00_g0f406df6.txt



In [ ]:
if IN_COLAB:
    !curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
    !./bin/micromamba create -y -n igwn -c conda-forge lalpulsar

In [ ]:
## check to see that lalpulsar_MakeSFTs is installed

!./bin/micromamba run -n igwn which lalpulsar_MakeSFTs

In [ ]:
sys.path.append(os.path.abspath("../create_sfdbs"))
from download_all_data_from_run import download_gwf
from make_ffl import *

In [ ]:
obs_run = 'O4a'
ifo = 'H1'
runn = obs_run+'_4KHZ_R1'
channel = ifo+':GWOSC-4KHZ_R1_STRAIN'#obs_run+'_4KHZ_R1'
save_dir = './data/'+obs_run+'/'+ifo+'/'
gps_start = 1369185055
gps_end = gps_start+1*4096

### Download the desired gravitaiotnal-wave frame (.gwf) files

In [ ]:
download_gwf(runn,ifo,save_dir,gps_start,gps_end)

In [ ]:
ffl_name = ifo+'_'+channel+'.ffl'
make_ffl(save_dir,ffl_name,absolute=True)

In [ ]:
def make_cache_file(ffl, ifo, channel_id, cache_out):
    site = ifo[0].upper()  # H1 -> H, L1 -> L, V1 -> V
    pat = re.compile(rf"^{site}-.*-(\d+)-(\d+)\.gwf$")  # capture gps, dur

    lines = []
    for p in Path(ffl).read_text().splitlines():
        p = p.strip()
        if not p or p.startswith("#"):
            continue
        gwf = Path(p).expanduser().resolve()
        m = pat.match(gwf.name)
        if not m:
            raise ValueError(f"Can't parse gps/dur from filename: {gwf.name}")
        gps, dur = m.group(1), m.group(2)
        lines.append(f"{site} {channel_id} {gps} {dur} file://localhost{gwf}")

    Path(cache_out).write_text("\n".join(lines) + "\n")
#     return str(Path(cache_out).resolve())


cache_fname = ifo+'_'+obs_run+'.cache'

make_cache_file(ffl=ffl_name,ifo=ifo,channel_id=channel,cache_out=cache_fname)


In [ ]:
def run_make_sfts(
    cache_file,
    gps_start_time,
    gps_end_time,
    start_freq,
    band,
    channel_name,
    high_pass_freq=7.0,
    sft_duration=1800,
    sft_write_path="./",
    observing_run=4,
    observing_kind="DEV",
    observing_revision=1,
    window="tukey",
    r=0.001,
    frame_checksums=True,
    dry_run=False,
    # Colab/micromamba settings:
    use_micromamba=True,
    micromamba_exe="./bin/micromamba",
    micromamba_env="igwn",
    make_sfts_exe="lalpulsar_MakeSFTs",
):
    cache_file = str(Path(cache_file).expanduser().resolve())
    if not Path(cache_file).exists():
        raise FileNotFoundError(f"Cache file not found: {cache_file}")

    outdir = Path(sft_write_path).expanduser()
    outdir.mkdir(parents=True, exist_ok=True)

    # Base command: either run inside micromamba env or call binary directly
    if use_micromamba:
        mm = Path(micromamba_exe)
        if not mm.exists():
            raise FileNotFoundError(
                f"micromamba not found at {mm}. "
                "Did you install it (curl | tar ...)?"
            )
        cmd = [str(mm), "run", "-n", micromamba_env, make_sfts_exe]
    else:
        # If you previously injected PATH in the kernel, this can work
        cmd = [make_sfts_exe]

    # Append MakeSFTs arguments
    cmd += [
        "--frame-cache", cache_file,
        "--frame-checksums", "TRUE" if frame_checksums else "FALSE",
        "--high-pass-freq", str(float(high_pass_freq)),
        "--sft-duration", str(int(sft_duration)),
        "--gps-start-time", str(int(gps_start_time)),
        "--gps-end-time", str(int(gps_end_time)),
        "--sft-write-path", str(outdir),
        "--start-freq", str(float(start_freq)),
        "--band", str(float(band)),
        "--channel-name", str(channel_name),
        "--observing-run", str(int(observing_run)),
        "--observing-kind", str(observing_kind),
        "--observing-revision", str(int(observing_revision)),
        "-w", str(window),
        "-r", str(float(r)),
    ]

    printable = " ".join(shlex.quote(x) for x in cmd)
    print(printable)

    if dry_run:
        return
    else:
        subprocess.run(cmd, check=True)

        sfts = sorted(outdir.glob("*.sft*"))
        print(f"\nWrote {len(sfts)} SFT file(s) to: {outdir.resolve()}")
        for p in sfts[:10]:
            print("  ", p.name)

#     return printable


In [ ]:
band = 100.0
start_freq = 100.0
outdir = "./sfts/"+obs_run+'/'+ifo+'/'
run_make_sfts(
    cache_file=cache_fname,
    gps_start_time=gps_start,
    gps_end_time=gps_end,
    sft_write_path=outdir,
    start_freq=start_freq,
    band=band,
    channel_name=channel
)
